In [22]:
import cv2
import numpy as np


# ENHANCEMENT PIPELINE
def estimate_blur(image, threshold=200):
    laplacian = cv2.Laplacian(image, cv2.CV_64F)
    variance = laplacian.var()
    return variance, variance < threshold

def estimate_noise(image):
    blurred = cv2.GaussianBlur(image, (5, 5), 1.5)
    diff = cv2.absdiff(image, blurred)
    noise_level = np.mean(diff)
    return noise_level, noise_level > 10

def estimate_contrast(image):
    std_dev = np.std(image)
    hist = cv2.calcHist([image], [0], None, [256], [0, 256])
    non_zero = np.where(hist > 0)[0]
    if len(non_zero) > 0:
        spread = non_zero[-1] - non_zero[0]
    else:
        spread = 0
    low_contrast = (std_dev < 40) or (spread < 150)
    return std_dev, spread, low_contrast

def dynamic_preprocess(image):
    processed = image.copy()
    steps_applied = []
    
    blur_value, is_blurry = estimate_blur(image)
    if is_blurry:
        kernel = np.array([[-1,-1,-1],
                          [-1, 9,-1],
                          [-1,-1,-1]])
        processed = cv2.filter2D(processed, -1, kernel)
        steps_applied.append("Sharpening")
    
    noise_level, is_noisy = estimate_noise(processed)
    if is_noisy:
        if noise_level > 20:
            kernel_size = (7, 7)
        elif noise_level > 15:
            kernel_size = (5, 5)
        else:
            kernel_size = (3, 3)
        processed = cv2.GaussianBlur(processed, kernel_size, 1.0)
        steps_applied.append(f"Gaussian blur {kernel_size}")
    
    std_dev, spread, low_contrast = estimate_contrast(processed)
    if low_contrast:
        if is_noisy or is_blurry:
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            processed = clahe.apply(processed)
            steps_applied.append("CLAHE equalization")
        else:
            processed = cv2.equalizeHist(processed)
            steps_applied.append("Global histogram equalization")
    
    return processed, steps_applied


# SEGMENTATION FUNCTIONS
def segment_pothole_fused(image):
    h, w = image.shape
    blurred = cv2.GaussianBlur(image, (21, 21), 0)
    thresh = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 101, 12
    )
    kernel_small = np.ones((5, 5), np.uint8)
    opened = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_small, iterations=2)
    contours, _ = cv2.findContours(opened, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    
    fat_canvas = np.zeros_like(image)
    for cnt in contours:
        length = cv2.arcLength(cnt, closed=False)
        if length > 50:
            x, y, bw, bh = cv2.boundingRect(cnt)
            cx, cy = x + bw / 2, y + bh / 2
            if (w * 0.2 < cx < w * 0.8) and (h * 0.15 < cy < h * 0.75):
                cv2.drawContours(fat_canvas, [cnt], -1, 255, thickness=40)
    
    temp_mask = np.zeros_like(image)
    fill_contours, _ = cv2.findContours(fat_canvas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if fill_contours:
        main_blob = max(fill_contours, key=cv2.contourArea)
        cv2.drawContours(fat_canvas, [main_blob], -1, 255, -1)
        kernel_shrink = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41))
        temp_mask = cv2.erode(fat_canvas, kernel_shrink, iterations=1)
    
    final_mask = np.zeros_like(image)
    final_contours, _ = cv2.findContours(temp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if final_contours:
        true_main_blob = max(final_contours, key=cv2.contourArea)
        cv2.drawContours(final_mask, [true_main_blob], -1, 255, -1)
    
    return final_mask

def segment_alligator(image):
    blurred = cv2.GaussianBlur(image, (5, 5), 0)
    kernel_bh = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
    blackhat = cv2.morphologyEx(blurred, cv2.MORPH_BLACKHAT, kernel_bh)
    _, binary = cv2.threshold(blackhat, 85, 255, cv2.THRESH_BINARY)
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (20, 20))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel_close, iterations=3)
    
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros_like(closed)
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 8000:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        aspect = w / float(h) if h > 0 else 0
        hull_area = cv2.contourArea(cv2.convexHull(cnt))
        solidity = area / hull_area if hull_area > 0 else 0
        if (0.3 < aspect < 4.0) and (solidity > 0.4):
            cv2.drawContours(mask, [cnt], -1, 255, -1)
    return mask

def segment_precise_transverse(image):
    denoised = cv2.medianBlur(image, 11)
    edges = cv2.Canny(denoised, 80, 180)
    kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (45, 3))
    dilated = cv2.dilate(edges, kernel_h, iterations=2)
    closing_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 10))
    closed = cv2.morphologyEx(dilated, cv2.MORPH_CLOSE, closing_kernel)
    
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros_like(closed)
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 10000:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        aspect_ratio = w / float(h) if h > 0 else 0
        if aspect_ratio > 5.0:
            cv2.drawContours(mask, [cnt], -1, 255, -1)
    return mask


# MAIN CODE
base_path = "C:/Users/USER/Desktop/image processing project/image processing video/pothole 2/extracted/"
damage_type = "pothole"   # "pothole", "alligator", or "transverse"
start_index = 50
step = 5
delay_ms = 500            
max_frames = 100          

cv2.namedWindow("Segmentation Overlay", cv2.WINDOW_NORMAL)
cv2.namedWindow("Binary Mask", cv2.WINDOW_NORMAL)

count = start_index
frame_count = 0
pause = False

print("Controls: q = quit, p = pause/resume")
print("Focus on any OpenCV window, then press keys.")

while frame_count < max_frames:
    path = base_path + str(count) + ".jpg"
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"End of sequence at index {count}")
        break

    # 1. Enhance
    enhanced, _ = dynamic_preprocess(img)
    
    # 2. Segment
    if damage_type == 'pothole':
        mask = segment_pothole_fused(enhanced)
    elif damage_type == 'alligator':
        mask = segment_alligator(enhanced)
    else:  # transverse
        mask = segment_precise_transverse(enhanced)
    
    # 3. Create overlay
    overlay = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    overlay[mask > 0] = [0, 255, 0]          # green fill
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, (0, 255, 255), 2)  # cyan border
    
    # 4. Show windows
    cv2.imshow("Segmentation Overlay", overlay)
    cv2.imshow("Binary Mask", mask)

    
    # 5. Keyboard control
    key = cv2.waitKey(delay_ms if not pause else 0) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('p'):
        pause = not pause
        print("Paused" if pause else "Resumed")
    
    count += step
    frame_count += 1

cv2.destroyAllWindows()
print("Playback finished.")

Controls: q = quit, p = pause/resume
Focus on any OpenCV window, then press keys.
End of sequence at index 145
Playback finished.
